# Tự động hóa dự án đến kiểm tra chuyến đo

Phân vùng theo LEZ trước: A=thí điểm 2027; B=LEZ 2030 trừ A; C=ngoài LEZ 2030. Bộ 60 A20/B20/C20; bộ 30 A10/B8/C12. Bộ điều phối chạy cả kiểm tra độ nhạy mở rộng và dùng cache hợp lệ. Hướng dẫn: `work/docs/lez/PROJECT_AUTOMATION.md`.

T2 xuất hướng dẫn chạy, GPX và sơ đồ từ đúng chuỗi cung; xem mục 7.
T3 theo dõi lịch, điều kiện và tổ hợp khảo sát; xem mục 8.

T5 mở rộng chỉ tiêu vận hành; xem mục 9.

M0 chạy riêng ở [notebook 10](10_random_baseline.ipynb). [Notebook 11 — biểu đồ đối chứng](11_random_comparison_plots.ipynb) chỉ đọc M0 đã lưu để chia khoảng D và xem quan hệ D với độ phủ/chiều dài; không chạy lại bộ điều phối hoặc bốc mẫu. [Hướng dẫn](../docs/lez/RANDOM_COMPARISON_PLOTS.md).

In [1]:
from pathlib import Path
import sys, json, importlib
import pandas as pd
from IPython.display import display
candidates = [Path.cwd(), *Path.cwd().parents]
candidates += [p / "work" for p in candidates]
candidates += [Path.home() / "python" / "work"]
WORK = next((p.resolve() for p in candidates if (p / "scripts/lez/project_automation.py").is_file()), None)
if WORK is None:
    raise FileNotFoundError("Mở notebook từ folder work có scripts/lez/project_automation.py.")
if str(WORK / "scripts") not in sys.path:
    sys.path.insert(0, str(WORK / "scripts"))
from lez.common import input_path

import lez.project_automation as automation
importlib.reload(automation)
preflight, run, status = automation.preflight, automation.run, automation.status

## 1. Kiểm tra cấu hình và dữ liệu

Đọc snapshot, hash nguồn, quota, phiên bản chính sách, runtime PyQGIS và cấu hình độ nhạy. Cell này không dựng hình học hay gửi API; baseline độ nhạy được kiểm tra sau khi bước chọn 30 hoàn tất.


In [2]:
CHECK = preflight(WORK)
print(json.dumps(CHECK, ensure_ascii=False, indent=2))

{
  "status": "ready",
  "candidate_quota": {
    "A": 20,
    "B": 20,
    "C": 20
  },
  "selection_quota": {
    "A_inside": 10,
    "B_inside": 8,
    "C_outside": 12
  },
  "matrix_cells": 12,
  "snapshot_utc": "2026-10-01T00:00:00Z",
  "network_calls": 0,
  "normalization_directory": "work/data/processed/normalization_2030/f42f05664165446b93ad",
  "qgis_runtime": "/Users/quanh.hg/python/.pyqgis-runtime/bin/python3.12",
  "zone_basis": "LEZ2027_pilot_and_LEZ2030_policy_first",
  "legacy_normalization_role": "Hanoi_source_and_supplemental_audit_only",
  "target_reallocation": {
    "basis": "2027 pilot eligible routing availability, preserve 20A and original scoring/RC criteria",
    "A": {
      "RC2": 6,
      "RC3": 7,
      "RC4": 7
    },
    "B": {
      "RC1": 5,
      "RC2": 5,
      "RC3": 5,
      "RC4": 5
    },
    "C": {
      "RC1": 5,
      "RC2": 5,
      "RC3": 5,
      "RC4": 5
    }
  },
  "routing_unavailable_cells": [
    {
      "sampling_zone": "A",
      "rc

## 2. Chạy toàn bộ

60 ứng viên → 30 đề xuất → độ nhạy mở rộng → bản đồ tên đường → biểu mẫu → bằng chứng → thuộc tính/chấm lại/thay tuyến → hành lang có hướng → hướng dẫn T2/GPX → GPS/tốc độ/chỉ tiêu vận hành T5 (RPA/PKE/SAFD) → bảng tiến độ tổ hợp T3.

`force=False` dùng cache có hash hợp lệ. Độ nhạy giữ quota A10/B8/C12, không thay bộ 30 công bố. `OBSERVATIONS_CSV=None` và `MEASUREMENTS_DIR=None` dùng dữ liệu đã có và folder mặc định; thiếu dữ liệu thực địa sẽ báo chờ. Chưa xây dựng chu trình lái.


In [3]:
OBSERVATIONS_CSV = None
MEASUREMENTS_DIR = None
RESULT = run(WORK, force=False, progress=print,
             observations=OBSERVATIONS_CSV, input_dir=MEASUREMENTS_DIR)
print("Trạng thái:", RESULT["status"])
print("60/30:", RESULT["candidate_routes"], RESULT["selected_routes"])
print("Chuyến thực:", RESULT["measured_trips"])
print("Còn chờ:", RESULT["pending"])
print("Độ nhạy:", RESULT["sensitivity_completed_scenarios"], "/", RESULT["sensitivity_scenarios"],
      "; ca không đủ quota có chủ đích:", RESULT["sensitivity_expected_infeasible_scenarios"])

print("T3: tổ hợp có chuyến hợp lệ / tổng:", RESULT["survey_cells_with_valid_trips"], "/", RESULT["survey_planned_cells"])
print("T3:", RESULT["survey_coverage_status"])

print("T5:", RESULT.get("operating_metrics_status"), RESULT.get("operating_metrics_version"))


Đang xử lý: sampling_60
Đang xử lý: selection_30
Đang xử lý: selection_sensitivity
Đọc dữ liệu đã kiểm tra; tái lập baseline trước khi thử độ nhạy.
[1/114] baseline: completed; giữ 30/30; D=0.365187
[2/114] X_zscore: completed; giữ 30/30; D=0.365187
[3/114] weight_C1_minus20: completed; giữ 30/30; D=0.365187
[4/114] weight_C1_plus20: completed; giữ 30/30; D=0.365187
[5/114] weight_C2_minus20: completed; giữ 30/30; D=0.365187
[6/114] weight_C2_plus20: completed; giữ 30/30; D=0.365187
[7/114] weight_C3_minus20: completed; giữ 30/30; D=0.365187
[8/114] weight_C3_plus20: completed; giữ 30/30; D=0.365187
[9/114] weight_C4_minus20: completed; giữ 30/30; D=0.365187
[10/114] weight_C4_plus20: completed; giữ 30/30; D=0.365187
[11/114] weight_C5_minus20: completed; giữ 30/30; D=0.365187
[12/114] weight_C5_plus20: completed; giữ 30/30; D=0.365187
[13/114] weight_C6_minus20: completed; giữ 30/30; D=0.365187
[14/114] weight_C6_plus20: completed; giữ 30/30; D=0.365187
[15/114] weight_C7_minus20: com

## 3. Xem tiến độ và lỗi

Báo cáo lưu sau từng bước; `selection_sensitivity` là bước riêng. Thời gian trong bảng là thời gian thực của bước ở lượt điều phối này, gồm cả lượt dùng cache. Kịch bản chưa giải quyết chặn trạng thái hoàn tất; một ca kiểm tra cố ý thiếu quota được báo riêng.


In [4]:
REPORT = status(WORK)
display(pd.DataFrame([
    {"step": key, "status": value.get("status"), "cache_hit": value.get("cache_hit"),
     "seconds_this_run": REPORT.get("step_seconds", {}).get(key),
     "completed_scenarios": value.get("completed_scenarios"),
     "expected_infeasible": value.get("expected_infeasible_scenarios"),
     "total_scenarios": value.get("total_scenarios")}
    for key, value in REPORT.get("steps", {}).items()
]))
print(json.dumps(REPORT.get("steps", {}).get("field_status", {}), ensure_ascii=False, indent=2))
print(json.dumps(REPORT.get("steps", {}).get("real_trip_qc", {}), ensure_ascii=False, indent=2))
print("Trạng thái:", REPORT.get("status"))
print("Bước đang chạy hoặc bị ngắt:", REPORT.get("active_step"))
print("Lỗi lần chạy:", REPORT.get("error"))


,step,status,cache_hit,seconds_this_run,completed_scenarios,expected_infeasible,total_scenarios
0,sampling_60,geometry_quota_achieved_pending_validation,True,2.201514,NaN,NaN,NaN
1,selection_30,proposed_for_survey,True,0.288285,NaN,NaN,NaN
2,selection_sensitivity,completed_pre_survey_sensitivity,False,27.221652,113.0,1.0,114.0
3,named_maps,cached,True,0.259022,NaN,NaN,NaN
4,field_templates,prepared_awaiting_field_observations,None,0.751086,NaN,NaN,NaN
5,field_status,awaiting_field_observations,None,0.260659,NaN,NaN,NaN
6,update_and_reserve_plan,proposed_update_plan,None,10.967802,NaN,NaN,NaN
7,observed_features_and_reselection,awaiting_field_observations,None,0.537502,NaN,NaN,NaN
8,directed_trip_corridors,prepared,False,4.553921,NaN,NaN,NaN
9,route_instructions,exported_pending_field_review,False,4.871377,NaN,NaN,NaN


{
  "status": "awaiting_field_observations",
  "candidate_routes": 60,
  "selected_routes": 30,
  "stored_observation_rows": 0,
  "valid_observations": 0,
  "selected_routes_eligible_for_manual_review": 0,
  "selected_routes_pending": 30,
  "G1_G4_G5_G6_unknown_counts": {
    "G1": 30,
    "G4": 30,
    "G5": 30,
    "G6": 30
  },
  "C6_unknown_count": 30,
  "measurement_verified": false,
  "surveyed_set_locked": false,
  "source_selection_fingerprint": "78e0a76a51bcc2a17cce",
  "validation_errors": []
}
{
  "status": "awaiting_measurements",
  "is_test_data": false,
  "measurement_verified": false,
  "trip_count": 0,
  "point_count": 0,
  "route_count": 30,
  "input_files": {
    "trips": {
      "path": "/Users/quanh.hg/python/work/data/measurements/incoming/trips.csv",
      "exists": true,
      "sha256": "4ce73c4ead26f93f155d21aa7fdec3e809b70b297ee8548d1ce45e1fab2c0626"
    },
    "points": {
      "path": "/Users/quanh.hg/python/work/data/measurements/incoming/gps_points.csv",
  

## 4. Kết quả độ nhạy mở rộng

Đọc kết quả của bước độ nhạy trong lượt điều phối vừa chạy. So sánh min–max/z-score, trọng số, ngưỡng ô, làn/tốc độ thiếu, C6 khác nhau và giả lập mất tuyến. Xem riêng trước/sau hoán đổi; tần suất kịch bản không phải xác suất chọn. Ca C6=0 là giả lập, không phải quan sát thực địa.

Chi tiết: `work/docs/lez/SENSITIVITY.md`.


In [5]:
SENSITIVITY = REPORT.get("steps", {}).get("selection_sensitivity", {})
if not SENSITIVITY:
    print("Lượt điều phối chưa có bước độ nhạy mở rộng. Chạy lại mục 2.")
else:
    print(json.dumps({key: SENSITIVITY.get(key) for key in [
        "status", "cache_hit", "total_scenarios", "completed_scenarios",
        "expected_infeasible_scenarios", "unexpected_infeasible_scenarios",
        "baseline_reproduced_exactly", "scoring_stress_scenarios"
    ]}, ensure_ascii=False, indent=2))
    SENS_DIRECTORY = input_path(WORK, SENSITIVITY["paths"]["directory"])
    display(pd.read_csv(SENS_DIRECTORY / "families.csv", keep_default_na=False))
    display(pd.read_csv(SENS_DIRECTORY / "scenario_summary.csv", keep_default_na=False))
    display(pd.read_csv(SENS_DIRECTORY / "simulated_replacements.csv", keep_default_na=False))
    print("Kịch bản chưa giải quyết:", SENSITIVITY.get("unresolved", []))
    print("Báo cáo:", SENSITIVITY["paths"]["report"])


{
  "status": "completed_pre_survey_sensitivity",
  "cache_hit": false,
  "total_scenarios": 114,
  "completed_scenarios": 113,
  "expected_infeasible_scenarios": 1,
  "unexpected_infeasible_scenarios": 0,
  "baseline_reproduced_exactly": true,
  "scoring_stress_scenarios": 82
}


,family,total,completed,infeasible,unchanged,min_retained,max_replaced,min_jaccard,max_D,min_D
0,C6_adversarial,2,2,0,2,30,0,1.0,0.36518658692754424,0.3651865869275442
1,C6_policy,1,1,0,1,30,0,1.0,0.36518658692754424,0.36518658692754424
2,C6_random,32,32,0,32,30,0,1.0,0.36518658692754424,0.36518658692754424
3,availability,30,30,0,0,28,2,0.875,0.41742781463824075,0.36539906744175965
4,cell_threshold,2,2,0,2,30,0,1.0,0.36518658692754424,0.36518658692754424
5,combined_random,16,16,0,16,30,0,1.0,0.36518658692754424,0.36518658692754424
6,constant_rule,1,1,0,1,30,0,1.0,0.36518658692754424,0.36518658692754424
7,diagnostic,1,0,1,0,,,,,
8,imputation,6,6,0,6,30,0,1.0,0.36518658692754424,0.36518658692754424
9,missing_features,3,3,0,3,30,0,1.0,0.36518658692754424,0.36518658692754424


,scenario,family,status,reason,retained,replaced,jaccard,removed,added,D,...,unique_RC_coverage_fraction,material_cells_scenario_threshold,material_cells_common_30pct,quantile_groups_scenario_X,quantile_groups_common_9_features,max_unique_corridor_overlap,greedy_retained,swaps,quota_ok,seconds
0,baseline,baseline,completed,,30,0,1.0,,,0.36518658692754424,...,0.010741962737068794,11,11,33,33,0.5966994438350687,30,2,True,2.264933
1,X_zscore,scaling,completed,,30,0,1.0,,,0.36518658692754424,...,0.010741962737068794,11,11,33,33,0.5966994438350687,29,3,True,2.522800
2,weight_C1_minus20,weights,completed,,30,0,1.0,,,0.36518658692754424,...,0.010741962737068794,11,11,33,33,0.5966994438350687,27,4,True,2.201900
3,weight_C1_plus20,weights,completed,,30,0,1.0,,,0.36518658692754424,...,0.010741962737068794,11,11,33,33,0.5966994438350687,30,2,True,2.176574
4,weight_C2_minus20,weights,completed,,30,0,1.0,,,0.36518658692754424,...,0.010741962737068794,11,11,33,33,0.5966994438350687,28,3,True,2.605669
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109,unavailable_C13,availability,completed,,29,1,0.9354838709677419,C13,C14,0.3701653401475101,...,0.010596810645716211,11,11,33,33,0.5966994438350687,26,4,True,2.225170
110,unavailable_C16,availability,completed,,29,1,0.9354838709677419,C16,C14,0.3924391719328054,...,0.010329455919608594,11,11,33,33,0.5966994438350687,29,3,True,2.221323
111,unavailable_C19,availability,completed,,29,1,0.9354838709677419,C19,C14,0.3679863695787491,...,0.010366091618565406,11,11,33,33,0.5966994438350687,29,3,True,2.431768
112,unavailable_C20,availability,completed,,29,1,0.9354838709677419,C20,C14,0.38903780294050894,...,0.010466056265693396,11,11,33,33,0.5966994438350687,29,3,True,2.319546


,excluded,status,added,also_removed,D,reason
0,A01,completed,A14,,0.366279,
1,A02,completed,A14,,0.366989,
2,A03,completed,A14,,0.366828,
3,A06,completed,A14,,0.366915,
4,A08,completed,A14,,0.366085,
5,A09,completed,A14,,0.366002,
6,A10,completed,A14,,0.367407,
7,A11,completed,A14,,0.366027,
8,A17,completed,A14,,0.365682,
9,A18,completed,A14,,0.366848,


Kịch bản chưa giải quyết: []
Báo cáo: work/data/processed/selection_sensitivity/79a7ae78b56b133b286b/REPORT.md


## 5. Tiếp tục sau khi dừng hoặc sau khi có dữ liệu

Chạy lại cell ở mục 2 với cùng cấu hình. Sau Interrupt/Ctrl+C tại độ nhạy, module giữ checkpoint từng kịch bản; chạy lại tiếp tục bằng cache đã kiểm hash. Báo cáo ghi bước bị ngắt hoặc lỗi, giữ các bước đã xong. Không cần `force=True` để tiếp tục.


In [6]:
# Bỏ comment khi muốn tiếp tục:
# RESULT = run(WORK, force=False, progress=print,
#              observations=OBSERVATIONS_CSV, input_dir=MEASUREMENTS_DIR)
print("Báo cáo:", WORK / "reports/automation/latest_project_automation.json")

Báo cáo: /Users/quanh.hg/python/work/reports/automation/latest_project_automation.json


## 6. Các bản đồ và biểu mẫu

Bản đồ 30/60 và danh sách tên đường nằm trong `work/maps/routes`. Biểu mẫu/checklist và ưu tiên xác minh nằm trong bundle field_workflow. Ranh giới là proxy có nguồn và dải bất định; xác minh GIS chính thức và bằng chứng thực địa vẫn cần dữ liệu thực tế.


In [7]:
print("Bản đồ:", WORK / "maps/routes")
FIELD = REPORT.get("steps", {}).get("field_templates", {})
print("Biểu mẫu:", FIELD.get("directory"))
print("Notebook nhập quan sát: work/notebooks/07_field_workflow.ipynb")
print("Notebook dữ liệu đo: work/notebooks/08_process_measurements.ipynb")

Bản đồ: /Users/quanh.hg/python/work/maps/routes
Biểu mẫu: /Users/quanh.hg/python/work/data/processed/field_workflow/78e0a76a51bcc2a17cce
Notebook nhập quan sát: work/notebooks/07_field_workflow.ipynb
Notebook dữ liệu đo: work/notebooks/08_process_measurements.ipynb


## 7. Hướng dẫn chạy 30 tuyến (T2)

T2 giữ đúng thứ tự và chiều đi của các cung nguồn, kể cả đoạn đi lặp. Controller chạy bằng Python thường; worker PyQGIS chỉ lấy hình học và đổi tọa độ. Không gọi API hoặc chọn lại tuyến. Hướng dẫn: `work/docs/lez/ROUTE_INSTRUCTIONS.md`.


In [8]:
import lez.route_instructions as instructions
importlib.reload(instructions)
REVISION = status(WORK).get('steps', {}).get('observed_features_and_reselection', {})
T2_SELECTION_REPORT = 'reports/field/latest_survey_revision.json' if REVISION.get('surveyed_set_locked') else None
T2_CHECK = instructions.preflight(WORK, selection_report=T2_SELECTION_REPORT)
print(json.dumps(T2_CHECK, ensure_ascii=False, indent=2))


{
  "status": "ready",
  "route_count": 30,
  "directed_arc_occurrences": 4311,
  "routes": [
    "A01",
    "A02",
    "A03",
    "A06",
    "A08",
    "A09",
    "A10",
    "A11",
    "A17",
    "A18",
    "B02",
    "B04",
    "B08",
    "B12",
    "B16",
    "B17",
    "B19",
    "B20",
    "C03",
    "C04",
    "C05",
    "C06",
    "C07",
    "C08",
    "C11",
    "C12",
    "C13",
    "C16",
    "C19",
    "C20"
  ],
  "snapshot_utc": "2026-10-01T00:00:00Z",
  "config_sha256": "fba6912522c357a5468a93dd88dbb2061e45032609e7f4facf4966e342518176",
  "network_calls": 0,
  "source_modified": false
}


### 7.1. Xuất hoặc tiếp tục

Chạy lại cùng cell để dùng cache đã kiểm hash. Kết quả xuất xong vẫn báo chờ rà soát thực địa; không tự xác nhận biển báo, điểm dừng hoặc tên đường còn thiếu.


In [9]:
T2_RESULT = instructions.run(WORK, selection_report=T2_SELECTION_REPORT, force=False)
print('Tuyến / lượt đi qua đoạn / bước:', T2_RESULT['route_count'], T2_RESULT['arc_occurrences'], T2_RESULT['instruction_count'])
print('Trạng thái:', T2_RESULT['status'])
print('Mở bằng trình duyệt:', WORK / 'maps/route_instructions/index.html')


Tuyến / lượt đi qua đoạn / bước: 30 4311 3279
Trạng thái: exported_pending_field_review
Mở bằng trình duyệt: /Users/quanh.hg/python/work/maps/route_instructions/index.html


### 7.2. Xem tổng hợp và điểm cần rà soát

Các dòng cần rà soát là ghi chú về dữ liệu/hiện trường, không phải các tuyến tải hoặc sinh thất bại. Đơn vị một dòng trong bảng chi tiết là một bước, có thể gồm nhiều lượt đi qua cung.


In [10]:
T2_DIRECTORY = input_path(WORK, T2_RESULT['paths']['directory'])
display(pd.read_csv(T2_DIRECTORY / 'route_summary.csv', keep_default_na=False))
display(pd.read_csv(T2_DIRECTORY / 'review_points.csv', keep_default_na=False).head(30))
print(json.dumps(T2_RESULT['review_flag_counts'], ensure_ascii=False, indent=2))


,route_id,sampling_zone,group,length_m,arc_occurrences,instruction_count,review_steps,unnamed_length_m,start_latitude,start_longitude,end_latitude,end_longitude,route_version,source_bundle_sha256,field_verified
0,A01,A,inside,3013.878042,112,40,39,0.000000,21.029043,105.856554,21.025907,105.847195,651f62df52e08fdf9e5f0047f82ee574f3788f374d2138...,33d4badd0b534368b846d47633fa7ee7f17adc109ff041...,False
1,A02,A,inside,2669.786812,90,37,35,50.965372,21.025140,105.849863,21.026919,105.843759,5db7fd93e6c5777299e31e90b62b0c279dabcd902010dc...,33d4badd0b534368b846d47633fa7ee7f17adc109ff041...,False
2,A03,A,inside,2708.558868,90,35,35,0.000000,21.022368,105.856340,21.034242,105.845634,347618274b5ebb12a484d1ee2521c8be68d328bac3be80...,33d4badd0b534368b846d47633fa7ee7f17adc109ff041...,False
3,A06,A,inside,2740.884945,105,31,29,80.683763,21.020307,105.849861,21.024944,105.850536,26f7faaa7b874a3576916eb1501986cfa4c969db6686ff...,33d4badd0b534368b846d47633fa7ee7f17adc109ff041...,False
4,A08,A,inside,3374.437379,99,47,43,137.260842,21.037337,105.852235,21.019717,105.852032,fece9f7f095537c351061b24d7fef0eee3fbefc9d66f96...,33d4badd0b534368b846d47633fa7ee7f17adc109ff041...,False
5,A09,A,inside,3263.588371,118,44,41,95.673969,21.021397,105.857574,21.036291,105.847084,332cc6859e7dad59a39d369ea7c8092b60ee441998ac60...,33d4badd0b534368b846d47633fa7ee7f17adc109ff041...,False
6,A10,A,inside,2540.524164,99,43,42,0.000000,21.024725,105.858347,21.026188,105.846250,8d6734a192e23cbb1cdda69cc075893acb7a6a4581293e...,33d4badd0b534368b846d47633fa7ee7f17adc109ff041...,False
7,A11,A,inside,3154.087623,109,41,39,0.000000,21.019313,105.853271,21.022875,105.857478,64f61b518ccfb6d70be9dc45eea4d380e8121463c65fcf...,33d4badd0b534368b846d47633fa7ee7f17adc109ff041...,False
8,A17,A,inside,3542.365355,80,49,46,50.521771,21.023959,105.842510,21.033994,105.849936,c9c4b02e46c1565701f75dd20aec9e4d3d713a675deb77...,33d4badd0b534368b846d47633fa7ee7f17adc109ff041...,False
9,A18,A,inside,2697.578673,91,39,38,0.000000,21.021722,105.849762,21.024680,105.858225,0cbc41d0d3b339265b5647c0a2f4d8afaa72fa01b2536d...,33d4badd0b534368b846d47633fa7ee7f17adc109ff041...,False


,route_id,step_no,start_m,latitude,longitude,flag,reason
0,A01,1,0.000000,21.029043,105.856554,start_unverified,Điểm đầu theo tuyến nguồn; chưa xác minh chỗ d...
1,A01,2,38.216053,21.029063,105.856921,junction_review,Nút có nhiều cung ra trong mạng; kiểm tra nhán...
2,A01,3,191.079702,21.027737,105.857331,junction_review,Nút có nhiều cung ra trong mạng; kiểm tra nhán...
3,A01,4,298.332419,21.026808,105.857625,junction_review,Nút có nhiều cung ra trong mạng; kiểm tra nhán...
4,A01,5,315.195038,21.026662,105.857671,junction_review,Nút có nhiều cung ra trong mạng; kiểm tra nhán...
5,A01,6,370.269758,21.026183,105.857815,junction_review,Nút có nhiều cung ra trong mạng; kiểm tra nhán...
6,A01,7,448.405974,21.025506,105.858025,junction_review,Nút có nhiều cung ra trong mạng; kiểm tra nhán...
7,A01,8,589.824535,21.024518,105.857170,roundabout_review,"Vòng xuyến; đi theo nhánh được đánh dấu, chưa ..."
8,A01,9,609.070042,21.024480,105.856997,roundabout_review,"Vòng xuyến; đi theo nhánh được đánh dấu, chưa ..."
9,A01,9,609.070042,21.024480,105.856997,junction_review,Nút có nhiều cung ra trong mạng; kiểm tra nhán...


{
  "start_unverified": 30,
  "junction_review": 2982,
  "roundabout_review": 31,
  "end_unverified": 30,
  "unnamed_road": 1361,
  "sharp_turn_review": 6,
  "uturn_review": 1
}


## 8. Lịch và độ đầy đủ tổ hợp khảo sát (T3)
Có 30 tuyến × 3 khung giờ × 2 loại ngày = **180 tổ hợp**; lịch lặp 2 lần tạo **360 lượt dự kiến**. Số lượt này chưa phải yêu cầu bắt buộc. Mục tiêu chuyến/ngày hợp lệ mỗi tổ hợp để trống trong `work/config/field_workflow.json` đến khi chốt thiết kế.

Cell dưới đọc dữ liệu thực và kết quả kiểm tra hiện hành; không tự tạo chuyến. Nếu đổi CSV, chạy mục 2 hoặc notebook 08 trước. Hướng dẫn trường nhập và ngưỡng chất lượng: `work/docs/lez/SURVEY_COVERAGE.md`.


In [11]:
import lez.survey_coverage as survey_coverage
importlib.reload(survey_coverage)
T3_REVISION = status(WORK).get("steps", {}).get("observed_features_and_reselection", {})
T3_SELECTION_REPORT = "reports/field/latest_survey_revision.json" if T3_REVISION.get("surveyed_set_locked") else None
T3_MEASUREMENTS_DIR = MEASUREMENTS_DIR  # dùng cùng nguồn số đo đã chọn ở mục 2
T3 = survey_coverage.run(WORK, selection_report=T3_SELECTION_REPORT, input_dir=T3_MEASUREMENTS_DIR)
print(json.dumps({k: T3[k] for k in ["status", "design_status", "planned_slots", "planned_cells", "observed_cells", "coverage_eligible_trips", "reason_counts"]}, ensure_ascii=False, indent=2))
T3_DIRECTORY = Path(T3["paths"]["directory"])
display(pd.read_csv(T3_DIRECTORY / "coverage_matrix.csv", keep_default_na=False))


{
  "status": "awaiting_measurements",
  "design_status": "proposed",
  "planned_slots": 360,
  "planned_cells": 180,
  "observed_cells": 0,
  "coverage_eligible_trips": 0,
  "reason_counts": {}
}


,route_id,sampling_zone,lez2030_group,time_block,day_type,proposed_slots,attempted_trips,valid_trips,needs_correction_trips,distinct_valid_dates,...,minimum_valid_trips,minimum_distinct_dates,remaining_valid_trips,remaining_distinct_dates,target_met,status,unusual_event_valid_trips,weather_counts,route_version,source_bundle_sha256
0,A01,A,inside,morning_peak,weekday,2,0,0,0,0,...,,,,,,not_measured,0,{},651f62df52e08fdf9e5f0047f82ee574f3788f374d2138...,33d4badd0b534368b846d47633fa7ee7f17adc109ff041...
1,A01,A,inside,morning_peak,weekend,2,0,0,0,0,...,,,,,,not_measured,0,{},651f62df52e08fdf9e5f0047f82ee574f3788f374d2138...,33d4badd0b534368b846d47633fa7ee7f17adc109ff041...
2,A01,A,inside,day_off_peak,weekday,2,0,0,0,0,...,,,,,,not_measured,0,{},651f62df52e08fdf9e5f0047f82ee574f3788f374d2138...,33d4badd0b534368b846d47633fa7ee7f17adc109ff041...
3,A01,A,inside,day_off_peak,weekend,2,0,0,0,0,...,,,,,,not_measured,0,{},651f62df52e08fdf9e5f0047f82ee574f3788f374d2138...,33d4badd0b534368b846d47633fa7ee7f17adc109ff041...
4,A01,A,inside,evening_peak,weekday,2,0,0,0,0,...,,,,,,not_measured,0,{},651f62df52e08fdf9e5f0047f82ee574f3788f374d2138...,33d4badd0b534368b846d47633fa7ee7f17adc109ff041...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175,C20,C,outside,morning_peak,weekend,2,0,0,0,0,...,,,,,,not_measured,0,{},4a322856e592b6f6d650a12708de04a61e690cdf398b32...,33d4badd0b534368b846d47633fa7ee7f17adc109ff041...
176,C20,C,outside,day_off_peak,weekday,2,0,0,0,0,...,,,,,,not_measured,0,{},4a322856e592b6f6d650a12708de04a61e690cdf398b32...,33d4badd0b534368b846d47633fa7ee7f17adc109ff041...
177,C20,C,outside,day_off_peak,weekend,2,0,0,0,0,...,,,,,,not_measured,0,{},4a322856e592b6f6d650a12708de04a61e690cdf398b32...,33d4badd0b534368b846d47633fa7ee7f17adc109ff041...
178,C20,C,outside,evening_peak,weekday,2,0,0,0,0,...,,,,,,not_measured,0,{},4a322856e592b6f6d650a12708de04a61e690cdf398b32...,33d4badd0b534368b846d47633fa7ee7f17adc109ff041...


### 8.1. Lượt dự kiến, tổ hợp còn thiếu và chuyến cần sửa
`trip_schedule_template.csv` là lịch đề xuất, không phải đầu vào số đo. Ghi dữ liệu thực trong `trips.csv` và `gps_points.csv`. Báo cáo giữ riêng chuyến đo thử, ngày lễ và điều kiện bất thường. T3 kiểm độ phủ theo lịch; mức đủ dữ liệu và độ ổn định thuộc T6.


In [12]:
display(pd.read_csv(T3_DIRECTORY / "trip_schedule_template.csv", keep_default_na=False).head(20))
display(pd.read_csv(T3_DIRECTORY / "missing_or_incomplete_cells.csv", keep_default_na=False))
display(pd.read_csv(T3_DIRECTORY / "trip_coverage_audit.csv", keep_default_na=False))
display(pd.read_csv(T3_DIRECTORY / "condition_summary.csv", keep_default_na=False))
print("Vấn đề đầu vào:", T3["input_issues"])


,route_id,sampling_zone,policy_version,route_version,source_bundle_sha256,lez2030_group,trip_id,vehicle_id,driver_id,measurement_date,...,event_status,event_notes,collection_stage,time_block,proposed_local_start,proposed_local_end,planned_repeat,status,survey_design_version,design_status
0,A01,A,LEZ2027_2030_LEZ_FIRST_v1,651f62df52e08fdf9e5f0047f82ee574f3788f374d2138...,33d4badd0b534368b846d47633fa7ee7f17adc109ff041...,inside,,,,,...,,,main,morning_peak,07:00,09:00,1,schedule_template_not_measured,T3_weekday_weekend_proposal_v1,proposed
1,A01,A,LEZ2027_2030_LEZ_FIRST_v1,651f62df52e08fdf9e5f0047f82ee574f3788f374d2138...,33d4badd0b534368b846d47633fa7ee7f17adc109ff041...,inside,,,,,...,,,main,morning_peak,07:00,09:00,2,schedule_template_not_measured,T3_weekday_weekend_proposal_v1,proposed
2,A01,A,LEZ2027_2030_LEZ_FIRST_v1,651f62df52e08fdf9e5f0047f82ee574f3788f374d2138...,33d4badd0b534368b846d47633fa7ee7f17adc109ff041...,inside,,,,,...,,,main,morning_peak,07:00,09:00,1,schedule_template_not_measured,T3_weekday_weekend_proposal_v1,proposed
3,A01,A,LEZ2027_2030_LEZ_FIRST_v1,651f62df52e08fdf9e5f0047f82ee574f3788f374d2138...,33d4badd0b534368b846d47633fa7ee7f17adc109ff041...,inside,,,,,...,,,main,morning_peak,07:00,09:00,2,schedule_template_not_measured,T3_weekday_weekend_proposal_v1,proposed
4,A01,A,LEZ2027_2030_LEZ_FIRST_v1,651f62df52e08fdf9e5f0047f82ee574f3788f374d2138...,33d4badd0b534368b846d47633fa7ee7f17adc109ff041...,inside,,,,,...,,,main,day_off_peak,10:00,12:00,1,schedule_template_not_measured,T3_weekday_weekend_proposal_v1,proposed
5,A01,A,LEZ2027_2030_LEZ_FIRST_v1,651f62df52e08fdf9e5f0047f82ee574f3788f374d2138...,33d4badd0b534368b846d47633fa7ee7f17adc109ff041...,inside,,,,,...,,,main,day_off_peak,10:00,12:00,2,schedule_template_not_measured,T3_weekday_weekend_proposal_v1,proposed
6,A01,A,LEZ2027_2030_LEZ_FIRST_v1,651f62df52e08fdf9e5f0047f82ee574f3788f374d2138...,33d4badd0b534368b846d47633fa7ee7f17adc109ff041...,inside,,,,,...,,,main,day_off_peak,10:00,12:00,1,schedule_template_not_measured,T3_weekday_weekend_proposal_v1,proposed
7,A01,A,LEZ2027_2030_LEZ_FIRST_v1,651f62df52e08fdf9e5f0047f82ee574f3788f374d2138...,33d4badd0b534368b846d47633fa7ee7f17adc109ff041...,inside,,,,,...,,,main,day_off_peak,10:00,12:00,2,schedule_template_not_measured,T3_weekday_weekend_proposal_v1,proposed
8,A01,A,LEZ2027_2030_LEZ_FIRST_v1,651f62df52e08fdf9e5f0047f82ee574f3788f374d2138...,33d4badd0b534368b846d47633fa7ee7f17adc109ff041...,inside,,,,,...,,,main,evening_peak,17:00,19:00,1,schedule_template_not_measured,T3_weekday_weekend_proposal_v1,proposed
9,A01,A,LEZ2027_2030_LEZ_FIRST_v1,651f62df52e08fdf9e5f0047f82ee574f3788f374d2138...,33d4badd0b534368b846d47633fa7ee7f17adc109ff041...,inside,,,,,...,,,main,evening_peak,17:00,19:00,2,schedule_template_not_measured,T3_weekday_weekend_proposal_v1,proposed


,route_id,sampling_zone,lez2030_group,time_block,day_type,proposed_slots,attempted_trips,valid_trips,needs_correction_trips,distinct_valid_dates,...,minimum_valid_trips,minimum_distinct_dates,remaining_valid_trips,remaining_distinct_dates,target_met,status,unusual_event_valid_trips,weather_counts,route_version,source_bundle_sha256
0,A01,A,inside,morning_peak,weekday,2,0,0,0,0,...,,,,,,not_measured,0,{},651f62df52e08fdf9e5f0047f82ee574f3788f374d2138...,33d4badd0b534368b846d47633fa7ee7f17adc109ff041...
1,A01,A,inside,morning_peak,weekend,2,0,0,0,0,...,,,,,,not_measured,0,{},651f62df52e08fdf9e5f0047f82ee574f3788f374d2138...,33d4badd0b534368b846d47633fa7ee7f17adc109ff041...
2,A01,A,inside,day_off_peak,weekday,2,0,0,0,0,...,,,,,,not_measured,0,{},651f62df52e08fdf9e5f0047f82ee574f3788f374d2138...,33d4badd0b534368b846d47633fa7ee7f17adc109ff041...
3,A01,A,inside,day_off_peak,weekend,2,0,0,0,0,...,,,,,,not_measured,0,{},651f62df52e08fdf9e5f0047f82ee574f3788f374d2138...,33d4badd0b534368b846d47633fa7ee7f17adc109ff041...
4,A01,A,inside,evening_peak,weekday,2,0,0,0,0,...,,,,,,not_measured,0,{},651f62df52e08fdf9e5f0047f82ee574f3788f374d2138...,33d4badd0b534368b846d47633fa7ee7f17adc109ff041...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175,C20,C,outside,morning_peak,weekend,2,0,0,0,0,...,,,,,,not_measured,0,{},4a322856e592b6f6d650a12708de04a61e690cdf398b32...,33d4badd0b534368b846d47633fa7ee7f17adc109ff041...
176,C20,C,outside,day_off_peak,weekday,2,0,0,0,0,...,,,,,,not_measured,0,{},4a322856e592b6f6d650a12708de04a61e690cdf398b32...,33d4badd0b534368b846d47633fa7ee7f17adc109ff041...
177,C20,C,outside,day_off_peak,weekend,2,0,0,0,0,...,,,,,,not_measured,0,{},4a322856e592b6f6d650a12708de04a61e690cdf398b32...,33d4badd0b534368b846d47633fa7ee7f17adc109ff041...
178,C20,C,outside,evening_peak,weekday,2,0,0,0,0,...,,,,,,not_measured,0,{},4a322856e592b6f6d650a12708de04a61e690cdf398b32...,33d4badd0b534368b846d47633fa7ee7f17adc109ff041...


,trip_id,route_id,status,reasons,plan_id,device_id,travel_direction,day_type,calendar_checked,calendar_note,calendar_evidence_uri,weather,weather_notes,event_status,event_notes,collection_stage


,day_type,weather,event_status,collection_stage,trip_rows,quality_passed_trips,coverage_eligible_trips


Vấn đề đầu vào: []


## 9. Chỉ tiêu vận hành T5
Chạy lại dữ liệu thực bằng module nhập chuyến để tính RPA, PKE, SAFD và các chỉ tiêu đã có. Giữ nguyên nguồn, không tự tạo tốc độ/GPS. Chưa có số đo thì báo chờ. Cấu hình ở `work/config/trip_processing.json`; lưới và cách tích phân hiện là đề xuất có phiên bản. Hướng dẫn: `work/docs/lez/OPERATING_METRICS.md`.


In [13]:
import lez.operating_metrics as kinematics
import lez.trip_processing as trip_processor
importlib.reload(kinematics)
importlib.reload(trip_processor)
T5_CONFIG = trip_processor.load_config(WORK)
T5_REVISION = status(WORK).get("steps", {}).get("observed_features_and_reselection", {})
if T5_REVISION.get("surveyed_set_locked"):
    T5_CONFIG["selection_report"] = "reports/field/latest_survey_revision.json"
T5_MEASUREMENTS_DIR = MEASUREMENTS_DIR
T5_REAL = trip_processor.run(WORK, config=T5_CONFIG, input_dir=T5_MEASUREMENTS_DIR)
print(json.dumps({k: T5_REAL.get(k) for k in ["status", "trip_count", "point_count", "operating_metrics_status", "operating_metrics_version", "safd_trip_rows", "safd_group_rows"]}, ensure_ascii=False, indent=2))


{
  "status": "awaiting_measurements",
  "trip_count": 0,
  "point_count": 0,
  "operating_metrics_status": "awaiting_measurements",
  "operating_metrics_version": "T5_linear_interval_SI_v1",
  "safd_trip_rows": 0,
  "safd_group_rows": 0
}


### 9.1. Xem chỉ tiêu, SAFD và nhật ký
Tỷ số gộp tính từ tổng tử số/tổng mẫu số; không lấy trung bình tỷ số từng chuyến. Chuyến đo thử và khảo sát chính được giữ tách riêng. Nếu lưới không phủ hết dữ liệu, thời gian ngoài lưới vẫn được báo, không ép phần trong lưới thành 100%.


In [14]:
if T5_REAL.get("directory"):
    T5_DIRECTORY = Path(T5_REAL["directory"])
    display(pd.read_csv(T5_DIRECTORY / "operating_metrics.csv"))
    display(pd.read_csv(T5_DIRECTORY / "operating_summary.csv"))
    display(pd.read_csv(T5_DIRECTORY / "safd_by_group.csv").head(30))
    display(pd.read_csv(T5_DIRECTORY / "kinematic_intervals.csv").head(30))
else:
    print("Chờ chuỗi thời gian–tốc độ và thông tin chuyến thực; không có chỉ tiêu thực để hiển thị.")


Chờ chuỗi thời gian–tốc độ và thông tin chuyến thực; không có chỉ tiêu thực để hiển thị.


### 9.2. Kiểm tra công thức bằng ví dụ tính tay
Ví dụ giả kiểm tra đứng yên, chạy đều, tăng–giảm tốc, thời gian lấy mẫu không đều và khoảng trống. Báo cáo riêng có `is_test_data=true`, không ghi vào bộ số đo thực. Với cách tích phân mặc định, PKE = 2 × RPA; không coi hai chỉ số là bằng chứng độc lập cho T6.


In [15]:
T5_CHECK = kinematics.self_check(T5_CONFIG)
trip_processor.atomic_json(WORK / "reports/trips/t5_self_check.json", T5_CHECK)
display(pd.DataFrame(T5_CHECK["cases"]))
print("Kiểm tra công thức:", T5_CHECK["status"], "; dữ liệu giả:", T5_CHECK["is_test_data"])
print("Số chuyến thực được tạo:", T5_CHECK["real_measurements_created"])


,case,expected,actual,passed
0,stationary,"{'kinematic_valid_duration_s': 3, 'kinematic_d...","{'kinematic_valid_duration_s': 3.0, 'kinematic...",True
1,constant_speed,"{'kinematic_valid_duration_s': 3, 'kinematic_d...","{'kinematic_valid_duration_s': 3.0, 'kinematic...",True
2,accelerate_cruise_brake,"{'kinematic_valid_duration_s': 6, 'kinematic_d...","{'kinematic_valid_duration_s': 6.0, 'kinematic...",True
3,unequal_steps,"{'kinematic_valid_duration_s': 3, 'kinematic_d...","{'kinematic_valid_duration_s': 3.0, 'kinematic...",True
4,gap_then_recover,"{'kinematic_valid_duration_s': 1, 'kinematic_d...","{'kinematic_valid_duration_s': 1.0, 'kinematic...",True


Kiểm tra công thức: passed ; dữ liệu giả: True
Số chuyến thực được tạo: 0
